# 01 — Download MPI-LEMON Phenotypic Data

**Run on Kaggle:** New Notebook → paste this → Settings: GPU on, Internet on

This notebook downloads only the lightweight files from OpenNeuro `ds000221`:
- `participants.tsv` — phenotypic targets (STAI, NEO-FFI, TICS)
- Subject list (which subjects have resting-state fMRI)

The raw fMRI files (~300 MB each) are downloaded and deleted per-subject in notebook 02.

**After running:** Save output as a Kaggle Dataset named `lemon-meta`.
Go to the notebook output panel → click the output folder → Save Version → tick
"Save as Dataset" → name it `lemon-meta`.

In [ ]:
!pip install -q awscli nibabel

In [ ]:
import os, json, subprocess
import pandas as pd
import numpy as np

OUT_DIR = '/kaggle/working'
BUCKET  = 's3://openneuro.org/ds000221'
print('Output dir:', OUT_DIR)

In [ ]:
# ── Download participants.tsv ──────────────────────────────────────────────────
!aws s3 cp {BUCKET}/participants.tsv /kaggle/working/participants.tsv --no-sign-request

participants = pd.read_csv('/kaggle/working/participants.tsv', sep='\t')
print(f'Participants: {len(participants)}')
print(f'Columns: {participants.columns.tolist()}')
participants.head()

In [ ]:
# ── List all subjects available on S3 ─────────────────────────────────────────
result = subprocess.run(
    ['aws', 's3', 'ls', f'{BUCKET}/', '--no-sign-request'],
    capture_output=True, text=True
)
subjects = sorted([
    l.strip().rstrip('/').split()[-1]
    for l in result.stdout.strip().split('\n') if 'sub-' in l
])
print(f'Subjects on S3: {len(subjects)}')
print('First 5:', subjects[:5])

In [ ]:
# ── Find phenotypic target columns ────────────────────────────────────────────
TARGET_PATTERNS = {
    'trait_anxiety':  ['STAI_T', 'STAI_Trait', 'stai_t', 'trait_anxiety'],
    'chronic_stress': ['TICS_SSCS', 'TICS', 'tics_sscs', 'chronic_stress'],
    'neuroticism':    ['NEO_N', 'neo_n', 'neuroticism', 'NEO_FFI_N'],
}

found = {}
for name, patterns in TARGET_PATTERNS.items():
    for p in patterns:
        matches = [c for c in participants.columns if p.lower() in c.lower()]
        if matches:
            found[name] = matches[0]
            vals = participants[matches[0]].dropna().values
            print(f'  {name:20s} → "{matches[0]}"  (n={len(vals)}, mean={vals.mean():.1f})')
            break
    if name not in found:
        print(f'  {name:20s} → NOT FOUND')
        print(f'    All columns: {participants.columns.tolist()}')

with open('/kaggle/working/target_columns.json', 'w') as f:
    json.dump(found, f, indent=2)
print('\ntarget_columns.json saved ✓')

In [ ]:
# ── Keep only subjects with all three targets ──────────────────────────────────
target_cols = list(found.values())
valid = participants.dropna(subset=target_cols).copy()
valid_ids = set(valid['participant_id'].str.replace('sub-', '', regex=False))

final_subjects = [s for s in subjects if s.replace('sub-', '') in valid_ids]
print(f'Subjects with phenotypic data + fMRI: {len(final_subjects)}')

valid.to_csv('/kaggle/working/phenotypic.csv', index=False)
with open('/kaggle/working/subject_list.json', 'w') as f:
    json.dump(final_subjects, f, indent=2)

print('phenotypic.csv saved ✓')
print('subject_list.json saved ✓')

In [ ]:
# ── Test: download one subject to confirm S3 access works ────────────────────
import nibabel as nib

test_sub  = final_subjects[0]
bold_key  = f'{BUCKET}/{test_sub}/func/{test_sub}_task-rest_bold.nii.gz'
test_path = f'/kaggle/working/{test_sub}_test.nii.gz'

print(f'Downloading {test_sub}...')
!aws s3 cp {bold_key} {test_path} --no-sign-request

img = nib.load(test_path)
print(f'Shape: {img.shape}   TR: {img.header.get_zooms()[3]:.2f}s')
os.remove(test_path)  # clean up
print('S3 access confirmed ✓')

In [ ]:
print('='*55)
print('NOTEBOOK 01 COMPLETE')
print('='*55)
print(f'  Subjects ready: {len(final_subjects)}')
print(f'  Targets: {list(found.keys())}')
print()
print('FILES IN /kaggle/working (save these as dataset "lemon-meta"):')
for f in os.listdir('/kaggle/working'):
    path = f'/kaggle/working/{f}'
    size = os.path.getsize(path)
    print(f'  {f}  ({size/1024:.1f} KB)')
print()
print('NEXT STEPS:')
print('  1. Click Save Version (top right)')
print('  2. Tick "Save as Dataset" → name it "lemon-meta"')
print('  3. Open notebook 02 and attach lemon-meta as input dataset')